# Dicionario de keywords a partir da Comprehensive Rules

*(Feito com ClaudeAI: https://claude.ai/share/b3536ad5-de2f-4b3f-a788-ba1c9a17f7da)*

Constroi um dicionario `keyword -> {definition, raw, source, category}` a
partir do arquivo `.txt` da Comprehensive Rules do Magic: The Gathering,
para uso no enriquecimento semantico de cartas (busca em linguagem
natural).

Fontes usadas dentro do proprio arquivo de regras:
- **Secao 701 (Keyword Actions) e 702 (Keyword Abilities)**: definicao
  "oficial" de cada keyword, a mais confiavel e a que tem prioridade.
- **Glossario** (fim do documento): cobre termos gerais do jogo e serve de
  fallback para keywords sem entrada propria em 701/702.

Basta ajustar o caminho do arquivo na celula de configuracao e rodar tudo
(`Run All`).

## Configuracao

In [1]:
import re
import json

RULES_PATH  = "Dados/Raw/MTG_Comprehensive_Rules.txt"              # caminho do .txt da Comprehensive Rules
OUTPUT_PATH = "Dados/Etapa1_card_expanded/keyword_dictionary.json" # onde o dicionario final sera salvo

## Corrigir encoding

O arquivo da Wizards costuma vir com um bug classico de dupla-codificacao
UTF-8 (bytes UTF-8 foram lidos como latin-1 e regravados como UTF-8), o que
transforma aspas curvas e o simbolo (R) em sequencias tipo `â€™`. Corrigimos
decodificando uma vez como utf-8 e reencodando/decodificando.

In [2]:
def fix_mojibake(raw_bytes: bytes) -> str:
    text = raw_bytes.decode("utf-8", errors="ignore")
    try:
        fixed = text.encode("cp1252", errors="ignore").decode("utf-8", errors="ignore")
    except UnicodeError:
        fixed = text
    return fixed.replace("\r\n", "\n")


with open(RULES_PATH, "rb") as f:
    raw = f.read()

text = fix_mojibake(raw)
print(f"Arquivo carregado: {len(text)} caracteres")

Arquivo carregado: 948568 caracteres


## Isolar secoes do documento

`701. Keyword Actions`, `702. Keyword Abilities` e `Glossary` aparecem duas
vezes no arquivo: uma no sumario (inicio) e outra na secao real. Pegamos
sempre a **ultima ocorrencia** de cada cabecalho como inicio do trecho a
extrair.

In [3]:
def slice_section(text: str, start_heading: str, end_heading: str) -> str:
    starts = [m.start() for m in re.finditer(rf"^{re.escape(start_heading)}\s*$", text, re.MULTILINE)]
    ends = [m.start() for m in re.finditer(rf"^{re.escape(end_heading)}\s*$", text, re.MULTILINE)]
    start = starts[-1]
    end = next(e for e in ends if e > start)
    return text[start:end]


section_701 = slice_section(text, "701. Keyword Actions", "702. Keyword Abilities")
section_702 = slice_section(text, "702. Keyword Abilities", "703. Turn-Based Actions")
glossary_section = slice_section(text, "Glossary", "Credits")
print(f"Secao 701: {len(section_701)} caracteres")
print(f"Secao 702: {len(section_702)} caracteres")
print(f"Glossario: {len(glossary_section)} caracteres")

Secao 701: 58386 caracteres
Secao 702: 147090 caracteres
Glossario: 112212 caracteres


## Extrair a lista oficial de Ability Words (regra 207.2c)

A propria Comprehensive Rules lista todas as *ability words* numa unica
frase da regra 207.2c (`"The ability words are adamant, addendum, ..."`).
Ability words **nao tem entrada propria no dicionario** -- a regra explica
que elas "have no special rules meaning and no individual entries in the
Comprehensive Rules". Extraimos essa lista pra usar depois, na etapa de
juncao com as cartas, pra classificar automaticamente quais keywords sem
match sao esperadas (ability word, efeito ja esta no oracle text da carta)
e quais sao uma lacuna real do dicionario.

In [4]:
def extract_ability_words(text: str) -> set:
    m = re.search(r"The ability words are (.+?)\.", text)
    if not m:
        return set()
    words = re.split(r",\s*(?:and\s+)?", m.group(1))
    return {w.strip().lower() for w in words if w.strip()}


ability_words = extract_ability_words(text)
print(f"{len(ability_words)} ability words extraidas da regra 207.2c")
print(sorted(ability_words)[:10], "...")

61 ability words extraidas da regra 207.2c
['adamant', 'addendum', 'alliance', 'battalion', 'bloodrush', 'celebration', 'channel', 'chroma', 'cohort', 'constellation'] ...


## Limpeza de texto: referencias de regra e colchetes

Duas funcoes de limpeza, aplicadas so na versao `definition` (a versao
`raw` fica intacta, preservando tudo):

**`strip_rule_references`** remove referencias cruzadas tipo `(See rule
509, ...)`, `See rule 704.`, `See also rule 303.4.` -- e tambem o caso que
nao comeca com "see", tipo `"...as described in rule 702.19b"` ou `"an
exception to rule 704.5m"`, que passava batido nas versoes anteriores dessa
funcao.

**`strip_placeholder_brackets`** remove colchetes de notacao tipo
`Protection from [quality]` -- uma carta real nunca tem o colchete literal.
A auditoria apontou que remover colchetes nao e' necessariamente superior
pra todo caso de uso; por isso a versao `raw` os mantem, permitindo
comparar com/sem colchetes depois sem precisar reprocessar o arquivo
original.

In [5]:
def strip_rule_references(text: str) -> str:
    text = re.sub(r"\s*\(see (?:rules? )?\d[^()]*\)", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*\bsee (?:also )?(?:rules? )?\d.*?(?:\.(?!\d)|!|$)", "", text, flags=re.IGNORECASE)
    text = re.sub(
        r"\brules?\s+\d+(?:\.\d+[a-z]?)?(?:\s*(?:,|and|or)\s*\d+(?:\.\d+[a-z]?)?)*",
        "", text, flags=re.IGNORECASE,
    )
    text = re.sub(r"\s+([.,])", r"\1", text)
    return re.sub(r"\s{2,}", " ", text).strip()


def strip_placeholder_brackets(text: str) -> str:
    return re.sub(r"[\[\]]", "", text)

## Extrair keyword actions (701) e keyword abilities (702)

Cada entrada `701.N. Nome` / `702.N. Nome` e seguida por subregras
(`701.Na`, `702.Na`, ...) que contem a definicao de fato. As subregras
`701.1` e `702.1` sao paragrafos introdutorios (nao keywords) -- filtradas
pelo **numero da subregra ser exatamente "1"**, que e' uma convencao da CR
(a subregra `.1` de toda secao numerada e' sempre um paragrafo geral
introdutorio). Isso e' mais robusto que a heuristica anterior (tamanho do
nome > 30 caracteres), que so funcionava por coincidencia de nenhuma
keyword real ter um nome longo.

A primeira subregra costuma ser so uma categoria (`"Flying is an evasion
ability."`) — o efeito de verdade geralmente vem na segunda (`"can't be
blocked except by..."`). Por isso concatenamos todas as subregras, menos os
blocos de `Example:`, que sao so casos ilustrativos.

In [6]:
def parse_numbered_section(section_text: str, section_num: int) -> dict:
    header_pat = re.compile(rf"^{section_num}\.(\d+)\.\s+(.+)$", re.MULTILINE)
    matches = list(header_pat.finditer(section_text))

    entries = {}
    for i, m in enumerate(matches):
        subrule_num = m.group(1)
        name = m.group(2).strip()
        if subrule_num == "1":
            continue  # paragrafo introdutorio da secao, nao uma keyword

        body_start = m.end()
        body_end = matches[i + 1].start() if i + 1 < len(matches) else len(section_text)
        body = section_text[body_start:body_end].strip()

        paragraphs = re.split(r"\n\s*\n", body)
        cleaned = []
        for p in paragraphs:
            p = re.split(r"\nExample:", p)[0]
            p = re.sub(r"^\d+\.\d+[a-z]?\s*", "", p).strip()
            if p:
                cleaned.append(p)
        definition = " ".join(cleaned)
        definition = strip_rule_references(definition)
        definition = strip_placeholder_brackets(definition)

        entries[name] = {
            "definition": definition,
            "raw": body,
            "source": str(section_num),
            "category": "keyword_action" if section_num == 701 else "keyword_ability",
        }

        if name == "Cycling":
            for alias in ("Landcycling", "Islandcycling", "Swampcycling",
                          "Mountaincycling", "Plainscycling", "Forestcycling",
                          "Typecycling", "Basic Landcycling"):
                entries[alias] = entries[name]
    return entries


keyword_actions = parse_numbered_section(section_701, 701)
keyword_abilities = parse_numbered_section(section_702, 702)
print(f"{len(keyword_actions)} keyword actions extraidas (701)")
print(f"{len(keyword_abilities)} keyword abilities extraidas (702)")

69 keyword actions extraidas (701)
202 keyword abilities extraidas (702)


## Extrair o glossario

O glossario segue o padrao: termo em uma linha, definicao no(s)
paragrafo(s) seguinte(s), entradas separadas por linha em branco.

In [7]:
def parse_glossary(glossary_text: str) -> dict:
    glossary_text = re.sub(r"^Glossary\s*\n", "", glossary_text.strip())
    blocks = re.split(r"\n\s*\n", glossary_text)

    glossary = {}
    for block in blocks:
        lines = block.strip().split("\n")
        if not lines or not lines[0].strip():
            continue
        term_header = lines[0].strip()
        raw = " ".join(l.strip() for l in lines[1:]).strip()
        definition = strip_rule_references(raw)
        definition = strip_placeholder_brackets(definition)
        if not definition:
            continue

        entry = {
            "definition": definition,
            "raw": raw,
            "source": "glossary",
            "category": "glossary",
        }
        for variant in term_header.split(","):
            variant = re.sub(r"\s*\[[^\]]*\]", "", variant).strip()
            if variant:
                glossary[variant] = entry
    return glossary


glossary = parse_glossary(glossary_section)
print(f"{len(glossary)} termos de glossario extraidos")

753 termos de glossario extraidos


## Juntar tudo e salvar

As secoes 701/702 sao a fonte "oficial": quando um termo existe tambem no
glossario, a definicao extraida da secao numerada tem prioridade. Tambem
salvamos a lista de ability words num arquivo separado, pra usar na etapa
de juncao com as cartas.

In [8]:
dictionary = dict(glossary)
dictionary.update(keyword_actions)
dictionary.update(keyword_abilities)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(dictionary, f, ensure_ascii=False, indent=2)

ability_words_path = OUTPUT_PATH.rsplit(".", 1)[0] + "_ability_words.json"
with open(ability_words_path, "w", encoding="utf-8") as f:
    json.dump(sorted(ability_words), f, ensure_ascii=False, indent=2)

n_701 = sum(1 for v in dictionary.values() if v["source"] == "701")
n_702 = sum(1 for v in dictionary.values() if v["source"] == "702")
n_gl = sum(1 for v in dictionary.values() if v["source"] == "glossary")
print(f"Dicionario salvo em {OUTPUT_PATH}")
print(f"  {n_701} keyword actions (secao 701)")
print(f"  {n_702} keyword abilities (secao 702)")
print(f"  {n_gl} termos so no glossario")
print(f"  {len(dictionary)} termos no total")
print(f"  {len(ability_words)} ability words -> {ability_words_path}")

Dicionario salvo em Dados/Etapa1_card_expanded/keyword_dictionary.json
  69 keyword actions (secao 701)
  202 keyword abilities (secao 702)
  489 termos so no glossario
  760 termos no total
  61 ability words -> Dados/Etapa1_card_expanded/keyword_dictionary_ability_words.json


## Conferindo alguns exemplos

In [9]:
for term in ["Flying", "Menace", "Trample Over Planeswalkers"]:
    entry = dictionary[term]
    print(f"=== {term} (fonte: {entry['source']}, categoria: {entry['category']}) ===")
    print("definition:", entry["definition"])
    print()

=== Flying (fonte: 702, categoria: keyword_ability) ===
definition: Flying is an evasion ability. A creature with flying can’t be blocked except by creatures with flying and/or reach. A creature with flying can block a creature with or without flying. Multiple instances of flying on the same creature are redundant.

=== Menace (fonte: 702, categoria: keyword_ability) ===
definition: Menace is an evasion ability. A creature with menace can’t be blocked except by two or more creatures. Multiple instances of menace on the same creature are redundant.

=== Trample Over Planeswalkers (fonte: glossary, categoria: glossary) ===
definition: A variant of trample that modifies how a creature assigns combat damage if it’s attacking a planeswalker.

